# Load Datasets to Google Cloud Storage
- May 2026
- Numantic Solutions (numanticsolutions.com)

In [1]:
import os, sys
import json

# Pandas
import pandas as pd

# Numantic utilities
utils_path = "../utils"
sys.path.insert(0, utils_path)
from utils import ApiAuthentication
api_configs = ApiAuthentication(client="Numantic")

# Tools for loading data to GCS
import gcs_data_load as gdl


## Read local document and test Q & A data


In [2]:
input_data_path = "../data/rag_eval_dataset"
docs_filename = "documents.csv"
no_answer_qs = "no_answer_questions.csv"
single_pas_answer_qs = "single_passage_answer_questions.csv"

# Read local data into Pandas dataframes
df_docs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, docs_filename))
df_noaqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, no_answer_qs))
df_spqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, single_pas_answer_qs))

## Clean up documents

## Load some document metadata

In [3]:
doc_metadata_file ="doc_metadata.json"
data_path = "../data/rag_eval_dataset"

with open(os.path.join(data_path,doc_metadata_file), "r", encoding="utf-8") as file:
    doc_metadata = json.load(file)


## Add metadata attributes and supplement text as needed

In [4]:
# Metadata columns
df_docs["source_type"] = df_docs["source_url"].map(doc_metadata["source_type_map"])
df_docs["title"] = df_docs["source_url"].map(doc_metadata["source_title_map"])

# Missing text
for key in doc_metadata["text_additions"].keys():
    doc_index = int(key.replace("doc_",""))
    df_docs.loc[doc_index, "text"] = "{}\n\n{}".format(df_docs.loc[1, "text"],
                                                       doc_metadata["text_additions"][key])

In [9]:
# df_docs.head()
df_docs

for idx in df_docs.index:
    df_docs.loc[idx, "text"] = df_docs.loc[idx, "text"][:5000]

df_docs

,index,source_url,text,source_type,title
0,0,https://enterthegungeon.fandom.com/wiki/Bullet...,Bullet Kin\nBullet Kin are one of the most com...,gaming,Bullet Kin
1,1,https://www.dropbox.com/scl/fi/ljtdg6eaucrbf1a...,---The Paths through the Underground/Underdark...,gaming,The Paths through the Underground/Underdark
2,2,https://bytes-and-nibbles.web.app/bytes/stici-...,Semantic and Textual Inference Chatbot Interfa...,data_science,Semantic and Textual Inference Chatbot Interfa...
3,3,https://github.com/llmware-ai/llmware,llmware\n\nBuilding Enterprise RAG Pipelines w...,data_science,LLMware
4,4,https://docs.marimo.io/recipes.html,Recipes\nThis page includes code snippets or “...,recipes,Building Block Recipes
5,5,https://towardsdatascience.com/how-to-maximize...,How to Maximize Your Impact as a Data Scientis...,data_science,How to Maximize Your Impact as a Data Scientist
6,6,https://ec.europa.eu/commission/presscorner/de...,Why do we need to regulate the use of Artifici...,government,Why do we need to regulate the use of Artifici...
7,7,https://bg3.wiki/wiki/The_Emperor,The Emperor is a mind flayer who appears in Ba...,gaming,The Emperor
8,8,https://whattocook.substack.com/p/so-into-nort...,so into northern spain!\nour magical urban-plu...,recipes,So into northern spain!
9,9,https://dmtalkies.com/the-zone-of-interest-end...,‘The Zone Of Interest’ Ending Explained & Film...,entertainment,The Zone Of Interest’ Ending Explained & Film ...


## Delete existing documents in GCS bucket

In [6]:

bucket_name = "ns_datasets"
doc_path = "rag-tests/documents/"


In [7]:
gdl.delete_existing_blobs(bucket_name=bucket_name,
                          key_prefix=doc_path)


Cleaned up 21 existing objects from rag-tests/documents/


## Save documents to Google Cloud Storage

In [8]:
gdl.process_and_upload_gcp(df=df_docs,
                           bucket_name=bucket_name,
                           text_col="text",
                           meta_cols=["source_url", "source_type", "title"],
                           key_prefix=doc_path)


Success: Uploaded 20 documents and manifest to rag-tests/documents/


In [25]:

def process_and_upload_gcp(df,
                           text_col: str,
                           meta_cols: list,
                           key_prefix='documents/'):
    metadata_records = []

    for idx in df.index:


        # Step 1: Get text
        raw_text = df.loc[idx, text_col]

        # Step 2: Get metadata fields
        meta_content = {}
        meta_content["document_index"] = f"doc_{idx}"
        for col in meta_cols:
            meta_content[col] = df.loc[idx, col]

        # Step 2: Annotate passage splits (keeping your logic)
        blocks = raw_text.split('\n\n')
        annotated_text = "".join([f"[{i}]\n{block}\n\n" for i, block in enumerate(blocks, 1)])

        # Step 3: Create filenames and GCS paths
        file_name = f"doc_{idx}.txt"
        blob_path = f"{key_prefix}{file_name}"
        gcs_uri = f"gs://{bucket_name}/{blob_path}"

        # Step 4: Upload Text File to GCS
        blob = bucket.blob(blob_path)
        blob.upload_from_string(annotated_text, content_type='text/plain')

        # Step 5: Create Vertex AI Metadata Entry
        # Note: Structuring 'structData' allows for advanced filtering in Vertex AI Search
        metadata_entry = {
            "id": meta_content["document_index"],
            "jsonData": json.dumps(meta_content),
            "content": {"mimeType": "text/plain", "uri": gcs_uri}
        }
        metadata_records.append(metadata_entry)

    # Step 6: Upload the consolidated metadata JSONL file
    # Vertex AI Search uses this file to import your data with filters
    metadata_blob = bucket.blob(f"{key_prefix}metadata.jsonl")
    metadata_jsonl = "\n".join([json.dumps(record) for record in metadata_records])
    metadata_blob.upload_from_string(metadata_jsonl, content_type='application/x-jsonlines')

    print(f"Uploaded {len(df)} docs and created metadata.jsonl at {key_prefix}")


In [28]:
# Initialize GCS client
storage_client = storage.Client()
bucket_name = 'ns_datasets'
bucket = storage_client.bucket(bucket_name)
doc_path = "rag-tests/documents/"

process_and_upload_gcp(df=df_docs,
                       key_prefix=doc_path)

Uploaded 20 docs and created metadata.jsonl at rag-tests/documents/
